In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("data/diabetic_data.csv")

In [5]:
df.shape

(101766, 50)

In [7]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [11]:
df.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

In [17]:
df['readmitted'].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [19]:
# Objective: Predict whether a diabetic patient will be readmitted to hospital

In [23]:
df.isnull().sum().sort_values(ascending=False).head(10)

max_glu_serum    96420
A1Cresult        84748
encounter_id         0
nateglinide          0
glimepiride          0
acetohexamide        0
glipizide            0
glyburide            0
tolbutamide          0
pioglitazone         0
dtype: int64

In [25]:
df[['max_glu_serum','A1Cresult']].head(10)

,max_glu_serum,A1Cresult
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN
6,NaN,NaN
7,NaN,NaN
8,NaN,NaN
9,NaN,NaN


In [27]:
df['readmitted'].value_counts(normalize=True)*100

readmitted
NO     53.911916
>30    34.928169
<30    11.159916
Name: proportion, dtype: float64

In [29]:
df['readmitted'].unique()

array(['NO', '>30', '<30'], dtype=object)

In [31]:
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 0 if x == 'NO' else 1)

df['readmitted_binary'].value_counts()

readmitted_binary
0    54864
1    46902
Name: count, dtype: int64

In [33]:
df = df.drop(columns=['max_glu_serum', 'A1Cresult'])

In [35]:
df.shape

(101766, 49)

In [37]:
df[['encounter_id', 'patient_nbr']].head()

,encounter_id,patient_nbr
0,2278392,8222157
1,149190,55629189
2,64410,86047875
3,500364,82442376
4,16680,42519267


In [39]:
print(df['encounter_id'].nunique())
print(df['patient_nbr'].nunique())

101766
71518


In [41]:
df = df.drop(columns=['encounter_id', 'patient_nbr'])

In [43]:
df.shape

(101766, 47)

In [45]:
df.isnull().sum().sort_values(ascending=False).head(10)

race             0
examide          0
glipizide        0
glyburide        0
tolbutamide      0
pioglitazone     0
rosiglitazone    0
acarbose         0
miglitol         0
troglitazone     0
dtype: int64

In [47]:
df['race'].value_counts(dropna=False)

race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

In [49]:
(df == '?').sum().sort_values(ascending=False).head(10)

weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
examide                  0
pioglitazone             0
rosiglitazone            0
dtype: int64

In [51]:
df['medical_specialty'].value_counts(dropna=False).head(10)

medical_specialty
?                             49949
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Name: count, dtype: int64

In [53]:
df['medical_specialty'].nunique()

73

In [55]:
df['age'].value_counts().sort_index()

age
[0-10)        161
[10-20)       691
[20-30)      1657
[30-40)      3775
[40-50)      9685
[50-60)     17256
[60-70)     22483
[70-80)     26068
[80-90)     17197
[90-100)     2793
Name: count, dtype: int64

In [57]:
df['payer_code'].nunique()

18

In [59]:
df['payer_code'].value_counts(dropna=False).head(10)

payer_code
?     40256
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2533
UN     2448
CM     1937
OG     1033
Name: count, dtype: int64

In [61]:
df['gender'].value_counts(dropna=False)

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

In [63]:
df['admission_type_id'].value_counts().sort_index()

admission_type_id
1    53990
2    18480
3    18869
4       10
5     4785
6     5291
7       21
8      320
Name: count, dtype: int64

In [65]:
df.select_dtypes(include='object').columns.tolist()

['race',
 'gender',
 'age',
 'weight',
 'payer_code',
 'medical_specialty',
 'diag_1',
 'diag_2',
 'diag_3',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

In [67]:
len(df.select_dtypes(include='object').columns)

35

In [69]:
df['weight'].value_counts(dropna=False)

weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

In [71]:
df['change'].value_counts()

change
No    54755
Ch    47011
Name: count, dtype: int64

In [73]:
round((df['weight'] == '?').mean()*100, 2)

96.86

In [75]:
round((df['payer_code'] == '?').mean()*100, 2)

39.56

In [79]:
df = df.drop(columns=['weight'])

KeyError: "['weight'] not found in axis"

In [81]:
df.shape

(101766, 46)

In [83]:
round((df['medical_specialty'] == '?').mean()*100, 2)

49.08

In [85]:
df['medical_specialty'].nunique()

73

In [87]:
df['medical_specialty'].value_counts().head(15)

medical_specialty
?                                  49949
InternalMedicine                   14635
Emergency/Trauma                    7565
Family/GeneralPractice              7440
Cardiology                          5352
Surgery-General                     3099
Nephrology                          1613
Orthopedics                         1400
Orthopedics-Reconstructive          1233
Radiologist                         1140
Pulmonology                          871
Psychiatry                           854
Urology                              685
ObstetricsandGynecology              671
Surgery-Cardiovascular/Thoracic      652
Name: count, dtype: int64

In [89]:
(df['medical_specialty'].value_counts() < 100).sum()

44

In [91]:
df[['diag_1','diag_2','diag_3']].head()

,diag_1,diag_2,diag_3
0,250.83,?,?
1,276,250.01,255
2,648,250,V27
3,8,250.43,403
4,197,157,250


In [93]:
(df[['diag_1','diag_2','diag_3']] == '?').sum()

diag_1      21
diag_2     358
diag_3    1423
dtype: int64

In [95]:
df[['diag_1','diag_2','diag_3']].describe()

,diag_1,diag_2,diag_3
count,101766,101766,101766
unique,717,749,790
top,428,276,250
freq,6862,6752,11555


In [97]:
df['diag_1'].value_counts().head(20)

diag_1
428      6862
414      6581
786      4016
410      3614
486      3508
427      2766
491      2275
715      2151
682      2042
434      2028
780      2019
996      1967
276      1889
38       1688
250.8    1680
599      1595
584      1520
V57      1207
250.6    1183
518      1115
Name: count, dtype: int64

In [99]:
import numpy as np

df[['diag_1','diag_2','diag_3']] = (
    df[['diag_1','diag_2','diag_3']]
    .replace('?', np.nan)
)

In [101]:
df[['diag_1','diag_2','diag_3']].isnull().sum()

diag_1      21
diag_2     358
diag_3    1423
dtype: int64

In [103]:
def categorize_diagnosis(diag):

    if pd.isna(diag):
        return 'Missing'

    diag = str(diag)

    if diag.startswith('V') or diag.startswith('E'):
        return 'Other'

    try:
        code = float(diag)

        if 390 <= code <= 459 or code == 785:
            return 'Circulatory'

        elif 460 <= code <= 519 or code == 786:
            return 'Respiratory'

        elif 520 <= code <= 579 or code == 787:
            return 'Digestive'

        elif 250 <= code < 251:
            return 'Diabetes'

        elif 800 <= code <= 999:
            return 'Injury'

        elif 710 <= code <= 739:
            return 'Musculoskeletal'

        elif 580 <= code <= 629 or code == 788:
            return 'Genitourinary'

        elif 140 <= code <= 239:
            return 'Neoplasms'

        else:
            return 'Other'

    except:
        return 'Other'

In [105]:
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col + '_cat'] = df[col].apply(categorize_diagnosis)

In [107]:
df[['diag_1',
    'diag_1_cat',
    'diag_2',
    'diag_2_cat',
    'diag_3',
    'diag_3_cat']].head(10)

,diag_1,diag_1_cat,diag_2,diag_2_cat,diag_3,diag_3_cat
0,250.83,Diabetes,NaN,Missing,NaN,Missing
1,276,Other,250.01,Diabetes,255,Other
2,648,Other,250,Diabetes,V27,Other
3,8,Other,250.43,Diabetes,403,Circulatory
4,197,Neoplasms,157,Neoplasms,250,Diabetes
5,414,Circulatory,411,Circulatory,250,Diabetes
6,414,Circulatory,411,Circulatory,V45,Other
7,428,Circulatory,492,Respiratory,250,Diabetes
8,398,Circulatory,427,Circulatory,38,Other
9,434,Circulatory,198,Neoplasms,486,Respiratory


In [109]:
for col in ['diag_1_cat', 'diag_2_cat', 'diag_3_cat']:
    print("\n", col)
    print(df[col].value_counts())


 diag_1_cat
diag_1_cat
Circulatory        30437
Other              18172
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Missing               21
Name: count, dtype: int64

 diag_2_cat
diag_2_cat
Circulatory        31881
Other              26553
Diabetes           12794
Respiratory        10895
Genitourinary       8376
Digestive           4170
Neoplasms           2547
Injury              2428
Musculoskeletal     1764
Missing              358
Name: count, dtype: int64

 diag_3_cat
diag_3_cat
Circulatory        30306
Other              29195
Diabetes           17157
Respiratory         7358
Genitourinary       6680
Digestive           3930
Injury              1946
Musculoskeletal     1915
Neoplasms           1856
Missing             1423
Name: count, dtype: int64


In [111]:
df['readmitted'].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [113]:
df['readmitted_binary'] = df['readmitted'].apply(
    lambda x: 1 if x == '<30' else 0
)

In [115]:
df['readmitted_binary'].value_counts()

readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 49 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      101766 non-null  object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   admission_type_id         101766 non-null  int64 
 4   discharge_disposition_id  101766 non-null  int64 
 5   admission_source_id       101766 non-null  int64 
 6   time_in_hospital          101766 non-null  int64 
 7   payer_code                101766 non-null  object
 8   medical_specialty         101766 non-null  object
 9   num_lab_procedures        101766 non-null  int64 
 10  num_procedures            101766 non-null  int64 
 11  num_medications           101766 non-null  int64 
 12  number_outpatient         101766 non-null  int64 
 13  number_emergency          101766 non-null  int64 
 14  numb

In [121]:
for col in [
    'metformin',
    'insulin',
    'change',
    'diabetesMed'
]:
    print("\n", col)
    print(df[col].value_counts())


 metformin
metformin
No        81778
Steady    18346
Up         1067
Down        575
Name: count, dtype: int64

 insulin
insulin
No        47383
Steady    30849
Down      12218
Up        11316
Name: count, dtype: int64

 change
change
No    54755
Ch    47011
Name: count, dtype: int64

 diabetesMed
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64


In [123]:
selected_features = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses'
]

df[selected_features].describe()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


In [127]:
df.to_csv("data/diabetic_data_processed.csv", index=False)

print("Processed dataset saved!")

Processed dataset saved!
